# Hallucination Detector — Evaluation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import glob, os

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='darkgrid', palette='muted')

In [ ]:
csv_files = sorted(glob.glob('results/results_*.csv'))
print(f'Found {len(csv_files)} file(s)')
latest = csv_files[-1]
df = pd.read_csv(latest)
print(f'Rows: {len(df)}')
df.head()

## 기본 통계

In [ ]:
valid = df[df['error'].isna() | (df['error'] == '')]
decided = valid[valid['verdict'] != 'UNCERTAIN'].copy()
decided['correct'] = (
    ((decided['verdict'] == 'HALLUCINATION') & (decided['label'] == 'hallucination')) |
    ((decided['verdict'] != 'HALLUCINATION') & (decided['label'] == 'factual'))
)
tp = ((decided['verdict']=='HALLUCINATION')&(decided['label']=='hallucination')).sum()
fp = ((decided['verdict']=='HALLUCINATION')&(decided['label']=='factual')).sum()
tn = ((decided['verdict']!='HALLUCINATION')&(decided['label']=='factual')).sum()
fn = ((decided['verdict']!='HALLUCINATION')&(decided['label']=='hallucination')).sum()
acc = decided['correct'].mean()
prec = tp/(tp+fp) if (tp+fp)>0 else 0
rec  = tp/(tp+fn) if (tp+fn)>0 else 0
f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
print(f'Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}')
print(f'TP={tp} FP={fp} TN={tn} FN={fn}')

## 혼동 행렬

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix((decided['label']=='hallucination').astype(int), (decided['verdict']=='HALLUCINATION').astype(int))
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Pred Factual','Pred Hallucination'],
    yticklabels=['Actual Factual','Actual Hallucination'], ax=ax)
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## Verdict 분포

In [ ]:
vc = valid['verdict'].value_counts()
colors = {'HALLUCINATION':'#ff5c5c','NO_HALLUCINATION':'#3de0a0','UNCERTAIN':'#ffb340','ERROR':'#888'}
fig, ax = plt.subplots(figsize=(6,3.5))
bars = ax.bar(vc.index, vc.values, color=[colors.get(v,'#aaa') for v in vc.index], width=0.5)
for b,v in zip(bars,vc.values): ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2, str(v), ha='center')
ax.set_title('Verdict Distribution')
sns.despine()
plt.tight_layout()
plt.show()

## 카테고리별 정확도

In [ ]:
cat = decided.groupby('category').agg(total=('correct','count'), correct=('correct','sum')).reset_index()
cat['acc'] = cat['correct']/cat['total']
cat = cat.sort_values('acc')
fig, ax = plt.subplots(figsize=(7, max(3, len(cat)*0.55)))
bars = ax.barh(cat['category'], cat['acc'], color='#5b8def', height=0.55)
ax.axvline(0.5, color='#ff5c5c', linewidth=1, linestyle='--', alpha=0.7)
for b,a,t in zip(bars,cat['acc'],cat['total']): ax.text(b.get_width()+0.01, b.get_y()+b.get_height()/2, f'{a:.0%} (n={t})', va='center', fontsize=10)
ax.set_xlim(0,1.2)
ax.set_title('Accuracy by Category')
sns.despine()
plt.tight_layout()
plt.show()

## Final Score 분포

In [ ]:
s = valid.copy()
s['final_score'] = pd.to_numeric(s['final_score'], errors='coerce')
fig, ax = plt.subplots(figsize=(7,4))
for lbl, color in [('hallucination','#ff5c5c'),('factual','#3de0a0')]:
    sub = s[s['label']==lbl]['final_score'].dropna()
    if len(sub)>1: sub.plot.kde(ax=ax, label=lbl, color=color, linewidth=2)
ax.axvline(0.5, color='#ffb340', linewidth=1.5, linestyle='--', label='t2=0.5')
ax.set_title('Final Score Distribution')
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

## 오답 케이스

In [ ]:
wrong = decided[~decided['correct']][['id','category','label','verdict','confidence','reason']]
print(f'Wrong: {len(wrong)}')
with pd.option_context('display.max_colwidth', 80): display(wrong)